<div dir="rtl">

# ⚡ 02 - FAISS Vector Store (Facebook AI Similarity Search)

## ما هو FAISS؟
- مكتبة مفتوحة المصدر طورتها شركة **Meta / Facebook AI** مخصصة للبحث فائق السرعة عن المتجهات المتشابهة (**Dense Vector Clustering & Approximate Nearest Neighbor Search**).
- مكتوبة بلغة C++ عالية الكفاءة مع واجهات Python لتقديم أقصى سرعة ممكنة.

---

### 🌟 أهم مميزات FAISS:
1. **سرعة خارقة**: مصممة للتعامل مع ملايين إلى مليارات المتجهات بكفاءة عالية.
2. **الحفظ والتحميل المحلي (Save & Load)**: حفظ الفهرس كملف ثنائي على القرص وتحميله لاحقاً دون إعادة حساب التضمينات.
3. **دمج الفهارس (Merge Indices)**: إمكانية دمج عدة فهارس منفصلة في فهرس واحد موحد (`merge_from`).
4. **تنوع مقاييس المسافة**: دعم مسافة L2 (Euclidean Distance) ومسافة Inner Product (Cosine Similarity).

</div>


<div dir="rtl">

### 1️⃣ استيراد المكتبات ونموذج التضمين

</div>


In [ ]:
import os
import shutil
from dotenv import load_dotenv, find_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

load_dotenv(find_dotenv())

# تهيئة نموذج التضمين
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ تم تحميل نموذج التضمين بنجاح!")


<div dir="rtl">

### 2️⃣ إنشاء الفهرس الأولي وإضافة المستندات

</div>


In [ ]:
docs_batch_1 = [
    Document(page_content="تعلم الآلة (Machine Learning) هو فرع من الذكاء الاصطناعي يركز على بناء خوارزميات تتعلم من البيانات.", metadata={"source": "ML_Guide"}),
    Document(page_content="الشبكات العصبية الالتفافية (CNN) ممتازة في معالجة الصور ومجالات الرؤية الحاسوبية.", metadata={"source": "CV_Notes"}),
    Document(page_content="معمارية Transformer أحدثت نقلة نوعية في معالجة اللغات الطبيعية ونماذج التوليد الحديثة.", metadata={"source": "NLP_Paper"})
]

# بناء فهرس FAISS من الدفعة الأولى
faiss_db1 = FAISS.from_documents(docs_batch_1, embeddings)
print("✅ تم إنشاء فهرس FAISS الأول بنجاح!")


<div dir="rtl">

### 3️⃣ البحث وحساب المسافات (L2 Distance Interpretation)
في FAISS الافتراضي (L2 Distance)، كلما كانت قيمة `score` **أقل (أقرب إلى 0)**، كلما كان التطابق الدلالي **أقوى وأقرب**.

</div>


In [ ]:
query = "ما هي أفضل النماذج لمعالجة الصور وفهمها؟"
results_with_score = faiss_db1.similarity_search_with_score(query, k=2)

print(f"🔍 الاستعلام: {query}\n")
for doc, score in results_with_score:
    print(f"المسافة (L2 Distance): {score:.4f} (أقل = أكثر شبهاً)")
    print(f"المحتوى: {doc.page_content}")
    print(f"المصدر: {doc.metadata['source']}")
    print("-" * 50)


<div dir="rtl">

### 4️⃣ حفظ الفهرس محلياً على القرص (Save Local)
يتم حفظ فهرس FAISS بصيغة ثنائية `index.faiss` وملف الميتاداتا `index.pkl`.

</div>


In [ ]:
save_path = "../../data/faiss_index"
os.makedirs(save_path, exist_ok=True)

# حفظ الفهرس على القرص
faiss_db1.save_local(save_path)
print(f"💾 تم حفظ فهرس FAISS بنجاح في: {os.path.abspath(save_path)}")
print("الملفات المحفوظة:", os.listdir(save_path))


<div dir="rtl">

### 5️⃣ إعادة تحميل الفهرس من القرص (Load Local)
نقوم بتحميل الفهرس من القرص واستخدامه مباشرة دون الحاجة لإعادة توليد التضمينات.
> **ملاحظة**: يجب تفعيل `allow_dangerous_deserialization=True` عند الثقة بمصدر ملف الـ pickle.

</div>


In [ ]:
loaded_faiss_db = FAISS.load_local(
    save_path,
    embeddings,
    allow_dangerous_deserialization=True
)

test_query = "أريد معرفة معمارية نماذج اللغة الحديثة"
loaded_result = loaded_faiss_db.similarity_search(test_query, k=1)

print("🎯 النتيجة من الفهرس المحمّل:")
print(f"المحتوى: {loaded_result[0].page_content}")
print(f"الميتاداتا: {loaded_result[0].metadata}")


<div dir="rtl">

### 6️⃣ دمج فهارس متعددة (Merge FAISS Indices)
ميزة قوية في FAISS تتيح دمج فهرس آخر في الفهرس الحالي دفعة واحدة دون إعادة بناء.

</div>


In [ ]:
# إنشاء فهرس ثانٍ بمستندات إضافية
docs_batch_2 = [
    Document(page_content="قواعد البيانات المتجهية مثل Chroma و Qdrant توفر قدرات فلترة متقدمة للـ RAG.", metadata={"source": "VectorDB_Guide"}),
    Document(page_content="نماذج التضمين مثل BGE و MiniLM تحول النصوص إلى أرقام تعكس المعنى السياقي بدقة.", metadata={"source": "Embeddings_Intro"})
]

faiss_db2 = FAISS.from_documents(docs_batch_2, embeddings)

# دمج الفهرس الثاني في الفهرس الأول
faiss_db1.merge_from(faiss_db2)

print("✅ تم دمج الفهرس الثاني في الأول بنجاح!")
merged_search = faiss_db1.similarity_search("كيف تعمل قواعد البيانات المتجهية؟", k=2)
for doc in merged_search:
    print(f"• {doc.page_content} (Source: {doc.metadata['source']})")
